In [3]:
import os
import subprocess
import time

BASE_DIR = os.getcwd()  # ✅ Notebook 里一定存在

def main(filename: str):
    mp3_dir = os.path.join(BASE_DIR, "outputs", "mp3")
    aac_dir = os.path.join(BASE_DIR, "static", "audio")

    mp3_path = os.path.join(mp3_dir, f"{filename}.mp3")
    aac_path = os.path.join(aac_dir, f"{filename}.m4a")

    if not os.path.isfile(mp3_path):
        raise FileNotFoundError(f"❌ MP3 不存在：{mp3_path}")

    os.makedirs(aac_dir, exist_ok=True)

    cmd = [
        "ffmpeg", "-y",
        "-i", mp3_path,
        "-c:a", "aac",
        "-b:a", "96k",
        "-ar", "44100",
        aac_path
    ]

    start = time.time()
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"✅ 已生成 AAC：{aac_path}")
    print(f"⏱️ 耗时：{time.time() - start:.2f}s")

main("Mt_1_en")

✅ 已生成 AAC：/Users/Clare/dev/syncbible/static/audio/Mt_1_en.m4a
⏱️ 耗时：2.29s


In [4]:
# 增量更新

import os
import subprocess
import time

BASE_DIR = os.getcwd()

MP3_DIR = os.path.join(BASE_DIR, "outputs", "mp3")
AAC_DIR = os.path.join(BASE_DIR, "static", "audio")

def batch_mp3_to_aac(mp3_dir: str, aac_dir: str, bitrate: str = "96k"):
    if not os.path.isdir(mp3_dir):
        raise FileNotFoundError(f"❌ 输入目录不存在：{mp3_dir}")

    os.makedirs(aac_dir, exist_ok=True)

    # 取文件名（不含扩展），做差集判断增量
    mp3_files = {
        f[:-4] for f in os.listdir(mp3_dir)
        if f.lower().endswith(".mp3") and os.path.isfile(os.path.join(mp3_dir, f))
    }
    aac_files = {
        f[:-4] for f in os.listdir(aac_dir)
        if f.lower().endswith(".m4a") and os.path.isfile(os.path.join(aac_dir, f))
    }

    to_convert = sorted(mp3_files - aac_files)
    skipped = len(mp3_files & aac_files)

    if skipped:
        print(f"⏭️  已存在跳过：{skipped} 个")
    if not to_convert:
        print("✅ 全部已同步，无需转换")
        return

    print(f"🎬 待转换 {len(to_convert)} 个：{to_convert}\n")

    total = len(to_convert)
    t_start = time.time()

    for i, name in enumerate(to_convert, 1):
        mp3_path = os.path.join(mp3_dir, f"{name}.mp3")
        aac_path = os.path.join(aac_dir, f"{name}.m4a")

        cmd = [
            "ffmpeg", "-y",
            "-i", mp3_path,
            "-c:a", "aac",
            "-b:a", bitrate,
            "-ar", "44100",
            aac_path
        ]
        s = time.time()
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        cost = time.time() - s

        print(f"[{i}/{total}] ✅ {name}.m4a  ({cost:.2f}s)")

    print(f"\n🏁 完成！共转换 {total} 个，总耗时 {time.time() - t_start:.2f}s")


if __name__ == "__main__":
    batch_mp3_to_aac(MP3_DIR, AAC_DIR, bitrate="96k")

⏭️  已存在跳过：1 个
🎬 待转换 3 个：['2K_4_en', 'Mt_10_en', 'Rom_6_en']



python(24068) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[1/3] ✅ 2K_4_en.m4a  (4.48s)


python(24073) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[2/3] ✅ Mt_10_en.m4a  (3.09s)


python(24080) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[3/3] ✅ Rom_6_en.m4a  (1.88s)

🏁 完成！共转换 3 个，总耗时 9.45s
